# Reinforcement Learning: From Bellman to Policy Gradients to RLHF
## Reward-driven learning and the mathematics of sequential decision making

**MAT 4953/6973 — Mathematical Foundations of AI** (Spring 2026, UTSA)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eduenez/MathAIspring2026UTSA/blob/main/code/reinforcement_learning.ipynb)

---

> **Prerequisites:** Probability (expectations, conditional distributions), calculus (gradients), familiarity with neural networks. No RL background assumed.

The supervised and unsupervised paradigms we have studied so far share a common structure: training data is given, and we minimise a loss derived from that data. **Reinforcement learning** (RL) is fundamentally different: an *agent* interacts with an *environment*, receives *rewards* as feedback, and must learn a *policy* — a mapping from situations to actions — that maximises long-run cumulative reward. There are no labelled examples; the agent must discover good behaviour through trial and error.

RL is the backbone of many landmark achievements (AlphaGo, game-playing AIs, robotic control) and, critically for this course, the fine-tuning stage that aligns large language models to human preferences (**RLHF**).

**Outline**
1. [The RL framework: MDPs](#part1)
2. [Value functions and the Bellman equation](#part2)
3. [Policy gradient methods](#part3)
4. [Deep RL and function approximation](#part4)
5. [RLHF: aligning language models](#part5)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from ipywidgets import interact, IntSlider, FloatSlider

plt.rcParams.update({'font.size': 12, 'axes.titlesize': 13, 'figure.dpi': 100})
rng = np.random.default_rng(0)
print("Libraries loaded.")

<a id="part1"></a>
# Part 1: The RL Framework — Markov Decision Processes

## 1.1 MDPs

A **Markov Decision Process** (MDP) is a tuple $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$:

| Symbol | Name | Description |
|--------|------|-------------|
| $\mathcal{S}$ | State space | Set of all possible situations the agent can be in |
| $\mathcal{A}$ | Action space | Set of all decisions the agent can make |
| $P(s' \mid s, a)$ | Transition dynamics | Probability of reaching $s'$ from $s$ after action $a$ |
| $R(s, a)$ | Reward function | Scalar signal received after taking action $a$ in state $s$ |
| $\gamma \in [0,1)$ | Discount factor | How much to value future rewards vs. immediate ones |

The agent's goal is to find a **policy** $\pi(a \mid s)$ — a distribution over actions given the current state — that maximises the **expected discounted return**:
$$J(\pi) = \mathbb{E}_\pi\!\left[\sum_{t=0}^{\infty} \gamma^t R(s_t, a_t)\right].$$

The **Markov property** is essential: the future depends only on the current state, not the full history. This property is what makes the Bellman equation tractable.

## 1.2 A gridworld MDP

We use a $5\times5$ gridworld as a concrete running example throughout this notebook.

In [ ]:
# ── Gridworld definition ─────────────────────────────────────────────────────
SIZE   = 5
N_S    = SIZE * SIZE                 # 25 states
N_A    = 4                           # up=0, down=1, left=2, right=3
GOAL   = N_S - 1                     # state 24 = cell (4,4)
WALLS  = {7, 11, 17}                 # blocked cells
DELTA  = [(-1,0),(1,0),(0,-1),(0,1)] # (dr, dc) for each action

def step(s, a):
    # Returns (s_next, reward)
    if s == GOAL: return s, 0.0
    r, c = divmod(s, SIZE)
    dr, dc = DELTA[a]
    rn, cn = max(0, min(SIZE-1, r+dr)), max(0, min(SIZE-1, c+dc))
    sn = rn * SIZE + cn
    if sn in WALLS: sn = s           # bounce off wall
    reward = 10.0 if sn == GOAL else -0.1
    return sn, reward

def draw_grid(ax, V=None, policy=None, title=''):
    ax.set_xlim(0, SIZE); ax.set_ylim(0, SIZE)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(title, fontsize=11)
    cmap = plt.cm.YlGn
    norm = Normalize(vmin=0, vmax=max(1.0, V.max()) if V is not None else 1.0)

    for s in range(N_S):
        r, c = divmod(s, SIZE)
        y, x = SIZE - 1 - r, c       # flip y for display
        if s in WALLS:
            ax.add_patch(plt.Rectangle((x, y), 1, 1, color='#555555'))
            continue
        fc = cmap(norm(V[s])) if V is not None else '#f0f0f0'
        ax.add_patch(plt.Rectangle((x, y), 1, 1, facecolor=fc,
                                    edgecolor='#aaaaaa', lw=0.8))
        if s == GOAL:
            ax.text(x+0.5, y+0.5, 'G', ha='center', va='center',
                    fontsize=13, fontweight='bold', color='#27ae60')
        elif s == 0:
            ax.text(x+0.5, y+0.5, 'S', ha='center', va='center',
                    fontsize=11, color='#2980b9')
        elif V is not None:
            ax.text(x+0.5, y+0.5, f'{V[s]:.1f}', ha='center', va='center',
                    fontsize=8, color='#333333')
        if policy is not None and s not in WALLS and s != GOAL:
            arrows = ['^','v','<','>']
            ax.text(x+0.5, y+0.5, arrows[policy[s]], ha='center', va='center',
                    fontsize=14, color='#c0392b')

fig, ax = plt.subplots(figsize=(4, 4))
draw_grid(ax, title='Gridworld  (S=start, G=goal, ■=wall)')
plt.tight_layout(); plt.show()

<a id="part2"></a>
# Part 2: Value Functions and the Bellman Equation

## 2.1 State-value function

The **state-value function** $V^\pi : \mathcal{S} \to \mathbb{R}$ measures how good it is to be in state $s$ when following policy $\pi$:
$$V^\pi(s) = \mathbb{E}_\pi\!\left[\sum_{t=0}^{\infty} \gamma^t R(s_t,a_t) \;\Big|\; s_0 = s\right].$$

The **Bellman equation** expresses $V^\pi$ as a fixed-point equation — the value of a state equals the expected immediate reward plus the discounted value of the next state:

$$\boxed{V^\pi(s) = \sum_{a} \pi(a\mid s)\left[R(s,a) + \gamma\sum_{s'} P(s'\mid s,a)\,V^\pi(s')\right].}$$

## 2.2 Optimal value and Bellman optimality

The **optimal value function** $V^*$ is achieved by the best possible policy:
$$\boxed{V^*(s) = \max_{a}\left[R(s,a) + \gamma\sum_{s'} P(s'\mid s,a)\,V^*(s')\right].}$$

This is the **Bellman optimality equation**. It is a fixed-point equation: $V^* = \mathcal{T} V^*$ where $\mathcal{T}$ is the Bellman optimality operator. Because $\mathcal{T}$ is a $\gamma$-contraction (in the $\ell^\infty$ norm), repeated application of $\mathcal{T}$ converges to the unique fixed point $V^*$.

**Value iteration** exploits this:

$$V_{k+1}(s) \leftarrow \max_{a}\left[R(s,a) + \gamma\sum_{s'} P(s'\mid s,a)\, V_k(s')\right], \qquad k = 0, 1, 2, \ldots$$

In [ ]:
def value_iteration(gamma=0.95, n_iter=60, eps=1e-6):
    V = np.zeros(N_S)
    policy = np.zeros(N_S, dtype=int)
    history = [V.copy()]

    for it in range(n_iter):
        V_new = V.copy()
        for s in range(N_S):
            if s == GOAL or s in WALLS:
                continue
            q_vals = []
            for a in range(N_A):
                sn, r = step(s, a)
                q_vals.append(r + gamma * V[sn])
            V_new[s]    = max(q_vals)
            policy[s]   = int(np.argmax(q_vals))
        history.append(V_new.copy())
        if np.max(np.abs(V_new - V)) < eps:
            print(f"Converged at iteration {it+1}")
            break
        V = V_new

    return V, policy, history

V_star, pi_star, V_history = value_iteration()

fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
iters_show = [0, 5, 20, len(V_history)-1]
for ax, it in zip(axes, iters_show):
    draw_grid(ax, V=V_history[it],
              policy=pi_star if it == len(V_history)-1 else None,
              title=f'Iteration {it}')
fig.suptitle(r'Value iteration: $V_k \to V^*$  (colour = value, arrows = optimal policy)',
             fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
def _plot_vi(iteration=0):
    it = min(iteration, len(V_history)-1)
    pol = pi_star if it == len(V_history)-1 else None
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    draw_grid(ax, V=V_history[it], policy=pol,
              title=f'Value iteration — step {it}')
    plt.tight_layout(); plt.show()

interact(_plot_vi, iteration=IntSlider(min=0, max=len(V_history)-1,
                                        step=1, value=0, description='Iter:'));

> **Exercise 2.1.** *(Bellman fixed point)*
>
> **(a)** Prove that the Bellman optimality operator $\mathcal{T}$, defined by $(\mathcal{T}V)(s) = \max_a[R(s,a) + \gamma\sum_{s'}P(s'|s,a)V(s')]$, is a $\gamma$-contraction in the $\ell^\infty$ norm: $\|\mathcal{T}V - \mathcal{T}W\|_\infty \le \gamma\|V-W\|_\infty$.
>
> **(b)** Use the contraction property to bound the error after $k$ iterations of value iteration: $\|V_k - V^*\|_\infty \le \gamma^k \|V_0 - V^*\|_\infty$. How many iterations are needed to reach error $\varepsilon = 0.01$ for $\gamma = 0.95$?
>
> **(c)** Modify the gridworld to add a second goal state with reward $+5$ at position $(0,4)$. Re-run value iteration. Does the optimal policy change?

<a id="part3"></a>
# Part 3: Policy Gradient Methods

## 3.1 Why not always use value iteration?

Value iteration requires knowing $P(s' \mid s, a)$ explicitly (a *model* of the environment) and storing a value for every state. For large or continuous state spaces (e.g., raw pixel inputs or physical sensor readings), both are infeasible.

**Policy gradient** methods directly parametrise and optimise the policy $\pi_\theta(a \mid s)$ — a neural network — without needing a model. They optimise $J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[G(\tau)]$ by gradient ascent.

## 3.2 The Policy Gradient Theorem

$$\boxed{\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\!\left[\sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t \mid s_t)\cdot G_t\right],}$$

where $G_t = \sum_{k=t}^{T}\gamma^{k-t}R(s_k,a_k)$ is the return from step $t$ onwards.

**Intuition:** $\nabla_\theta \log \pi_\theta(a_t \mid s_t)$ is the direction to increase the probability of action $a_t$; $G_t$ weights this by how good the subsequent trajectory was. Actions followed by high returns are reinforced; actions followed by low returns are suppressed.

The REINFORCE algorithm (Williams, 1992) uses Monte Carlo sampling to estimate this expectation:

```
Repeat:
    Collect episode τ = (s₀,a₀,r₀, s₁,a₁,r₁, ...) by rolling out π_θ
    For each t: compute Gₜ = Σ_{k≥t} γ^{k−t} rₖ
    Update θ ← θ + η · Σ_t ∇_θ log π_θ(aₜ|sₜ) · Gₜ
```

**Variance reduction:** $G_t$ has high variance. A common fix is to subtract a *baseline* $b(s_t)$ (often $V^\pi(s_t)$), giving the *advantage* $A_t = G_t - b(s_t)$. This is the actor-critic idea.

In [ ]:
# ── Simple REINFORCE on the gridworld ────────────────────────────────────────
# Policy: softmax over a linear function of a one-hot state encoding
# theta: (N_A, N_S) weight matrix

def policy_probs(theta, s):
    logits = theta[:, s]
    logits = logits - logits.max()   # numerical stability
    probs  = np.exp(logits); probs /= probs.sum()
    return probs

def collect_episode(theta, gamma=0.95, max_steps=200, rng_=None):
    if rng_ is None: rng_ = np.random.default_rng()
    s = 0; traj = []
    for _ in range(max_steps):
        probs = policy_probs(theta, s)
        a     = int(rng_.choice(N_A, p=probs))
        sn, r = step(s, a)
        traj.append((s, a, r))
        s = sn
        if s == GOAL: break
    return traj

def reinforce(n_episodes=3000, lr=0.02, gamma=0.95, seed=7):
    rng_ = np.random.default_rng(seed)
    theta = rng_.standard_normal((N_A, N_S)) * 0.01
    returns_log = []

    for ep in range(n_episodes):
        traj = collect_episode(theta, gamma, rng_=rng_)
        T_ep = len(traj)

        # Compute returns
        Gs = np.zeros(T_ep)
        G  = 0.0
        for t in reversed(range(T_ep)):
            G = traj[t][2] + gamma * G
            Gs[t] = G

        # Baseline: mean return (reduces variance)
        Gs = Gs - Gs.mean()

        # Gradient update
        for t, (s, a, _) in enumerate(traj):
            probs = policy_probs(theta, s)
            grad  = -probs.copy()
            grad[a] += 1.0                # ∇ log π(a|s) for softmax
            theta[:, s] += lr * grad * Gs[t]

        returns_log.append(sum(r for _, _, r in traj))

    return theta, returns_log

print("Training REINFORCE on gridworld …")
theta_rl, ret_log = reinforce(n_episodes=3000)
print("Done.")

# Smooth returns
window = 100
smoothed = np.convolve(ret_log, np.ones(window)/window, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ret_log, alpha=0.3, color='#3498db', lw=0.8)
axes[0].plot(np.arange(window-1, len(ret_log)), smoothed, color='#3498db', lw=2)
axes[0].axhline(0, color='k', lw=0.5, ls='--')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Episode return')
axes[0].set_title('REINFORCE learning curve')

V_rl = np.array([np.dot(policy_probs(theta_rl, s),
                         [step(s,a)[1] + 0.95 * V_star[step(s,a)[0]] for a in range(N_A)])
                  for s in range(N_S)])
draw_grid(axes[1], V=V_rl,
          policy=np.array([np.argmax([step(s,a)[1] + 0.95*V_star[step(s,a)[0]]
                                       for a in range(N_A)]) for s in range(N_S)]),
          title='Learned policy (REINFORCE)')
plt.tight_layout(); plt.show()

<a id="part4"></a>
# Part 4: Deep RL — Scaling Up with Neural Networks

The key step from classical RL to **deep RL** is replacing the tabular policy (a table with one entry per state) with a neural network $\pi_\theta$ or value function $V_\theta$. This allows generalisation across similar states and makes RL applicable to high-dimensional inputs like images.

**DQN** (Mnih et al., 2015): a deep network $Q_\theta(s, a)$ approximates the action-value function $Q^*(s,a) = R(s,a) + \gamma\max_{a'}Q^*(s',a')$. The network is trained by minimising the Bellman residual:
$$\mathcal{L}(\theta) = \mathbb{E}\!\left[\left(R + \gamma\max_{a'} Q_{\theta^-}(s',a') - Q_\theta(s,a)\right)^2\right],$$
where $\theta^-$ are *target network* parameters updated slowly. Two tricks stabilise training:
- **Experience replay**: store transitions $(s,a,r,s')$ in a buffer; sample random mini-batches.
- **Target network**: avoid moving target by delaying updates to $\theta^-$.

**Proximal Policy Optimisation (PPO)** (Schulman et al., 2017) is now the dominant algorithm for both continuous control and RLHF. It is an actor-critic method that clips the policy update:
$$\mathcal{L}^{\text{PPO}}(\theta) = \mathbb{E}\!\left[\min\!\left(r_t(\theta)\,A_t,\;\mathrm{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon)\,A_t\right)\right],$$
where $r_t(\theta) = \pi_\theta(a_t|s_t)/\pi_{\theta_{\text{old}}}(a_t|s_t)$ is the importance ratio and $A_t$ is the advantage. The clip prevents large destabilising policy updates — the "proximal" in PPO.

<a id="part5"></a>
# Part 5: RLHF — Reinforcement Learning from Human Feedback

## 5.1 The alignment problem

Large language models pre-trained on text are good at predicting tokens, but not necessarily at producing *helpful*, *harmless*, or *honest* responses. Supervised fine-tuning on curated examples helps but has limits — it is hard to specify every desirable behaviour. RLHF (Christiano et al., 2017; Ouyang et al., 2022) uses RL with a *learned reward model* trained on human preference judgements.

## 5.2 Three-stage pipeline

**Stage 1 — Supervised fine-tuning (SFT):** Fine-tune the base LLM on high-quality demonstrations to get $\pi_{\text{SFT}}$.

**Stage 2 — Reward model training:** Collect pairs of responses $(y_1, y_2)$ to the same prompt $x$ and ask human annotators: *"Which is better?"* Fit a scalar reward model $R_\phi(x, y)$ using the **Bradley–Terry** preference model:

$$P(y_1 \succ y_2 \mid x) = \sigma\!\left(R_\phi(x,y_1) - R_\phi(x,y_2)\right),$$
$$\mathcal{L}(\phi) = -\mathbb{E}_{(x,y_1,y_2)}\!\left[\log\sigma\!\left(R_\phi(x,y_1) - R_\phi(x,y_2)\right)\right].$$

**Stage 3 — RL fine-tuning:** Optimise the LLM policy $\pi_\theta$ (initialised from $\pi_{\text{SFT}}$) to maximise expected reward while staying close to the SFT baseline:

$$\boxed{\max_\theta \;\mathbb{E}_{x \sim \mathcal{D},\; y \sim \pi_\theta(\cdot|x)}\!\left[R_\phi(x,y)\right] - \lambda\, D_{\mathrm{KL}}\!\left(\pi_\theta(\cdot|x)\;\|\;\pi_{\mathrm{SFT}}(\cdot|x)\right).}$$

The **KL penalty** prevents the policy from "reward-hacking" — generating text that gets a high reward score but deviates so far from the SFT distribution that it becomes incoherent.

In [ ]:
# ── Visualise the reward–KL tradeoff ────────────────────────────────────────
# Synthetic illustration: reward increases then saturates; KL penalty grows

lam_values = [0.0, 0.1, 0.5, 2.0]
kl_range   = np.linspace(0, 10, 300)

def reward_fn(kl):
    # Reward saturates after some KL divergence (reward hacking regime)
    return 3.0 * (1 - np.exp(-0.8 * kl)) - 0.3 * np.maximum(0, kl - 4)**2

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
reward_curve = reward_fn(kl_range)
axes[0].plot(kl_range, reward_curve, color='#3498db', lw=2.5, label='Reward $R_\\phi$')
axes[0].set_xlabel(r'$D_\mathrm{KL}(\pi_\theta \| \pi_\mathrm{SFT})$')
axes[0].set_ylabel('Value')
axes[0].set_title('Reward model output vs. KL from SFT')
axes[0].legend(fontsize=11)

colors = ['#2ecc71', '#3498db', '#e74c3c', '#9b59b6']
for lam, c in zip(lam_values, colors):
    obj = reward_fn(kl_range) - lam * kl_range
    opt_idx = np.argmax(obj)
    axes[1].plot(kl_range, obj, color=c, lw=2, label=fr'$\lambda={lam}$')
    axes[1].axvline(kl_range[opt_idx], color=c, lw=0.8, ls='--', alpha=0.7)

axes[1].set_xlabel(r'$D_\mathrm{KL}(\pi_\theta \| \pi_\mathrm{SFT})$')
axes[1].set_ylabel(r'$\mathbb{E}[R_\phi] - \lambda\,D_\mathrm{KL}$')
axes[1].set_title(r'RLHF objective for different $\lambda$ values')
axes[1].legend(fontsize=10)

for ax in axes:
    ax.axhline(0, color='k', lw=0.4, ls='--')
    ax.grid(True, alpha=0.2)

plt.tight_layout(); plt.show()
print("Optimal KL for each λ:")
for lam, c in zip(lam_values, colors):
    obj = reward_fn(kl_range) - lam * kl_range
    print(f"  λ={lam:.1f}  →  KL* ≈ {kl_range[np.argmax(obj)]:.2f}")

In [ ]:
# ── Bradley–Terry reward model: likelihood landscape ─────────────────────────
# Illustrate: fitting R(x,y) from pairwise preferences

delta_r = np.linspace(-4, 4, 300)   # R(y1) - R(y2)
prob_y1_wins = 1 / (1 + np.exp(-delta_r))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(delta_r, prob_y1_wins, color='#3498db', lw=2.5)
axes[0].axhline(0.5, color='k', lw=0.6, ls='--')
axes[0].axvline(0.0, color='k', lw=0.6, ls='--')
axes[0].set_xlabel(r'$R_\phi(x,y_1) - R_\phi(x,y_2)$')
axes[0].set_ylabel(r'$P(y_1 \succ y_2)$')
axes[0].set_title('Bradley–Terry preference probability')
axes[0].fill_between(delta_r, 0.5, prob_y1_wins,
                      where=delta_r > 0, alpha=0.15, color='#2ecc71',
                      label=r'$y_1$ preferred')
axes[0].fill_between(delta_r, prob_y1_wins, 0.5,
                      where=delta_r < 0, alpha=0.15, color='#e74c3c',
                      label=r'$y_2$ preferred')
axes[0].legend(fontsize=10)

# Negative log-likelihood
nll = -np.log(prob_y1_wins + 1e-9)
axes[1].plot(delta_r, nll, color='#e74c3c', lw=2.5)
axes[1].set_xlabel(r'$R_\phi(x,y_1) - R_\phi(x,y_2)$')
axes[1].set_ylabel('Negative log-likelihood')
axes[1].set_title(r'Training loss: $-\log\sigma(\Delta R)$')
axes[1].set_ylim(0, 6)
axes[1].grid(True, alpha=0.2)

plt.tight_layout(); plt.show()

## 5.3 Constitutional AI and direct alignment

After RLHF, several variants have been proposed:

- **Direct Preference Optimisation (DPO)** (Rafailov et al., 2023) eliminates the explicit reward model by showing that the RLHF objective has a closed-form solution in terms of preference data, reducing training to supervised learning on comparisons.
- **Constitutional AI** (Anthropic, 2022) replaces human preference labels with a set of principles; the model self-critiques using those principles.

In all cases, the mathematical core is the same: a KL-regularised optimisation of a reward (or preference) signal.

> **Exercise 5.1.** *(Policy gradient derivation)*
>
> **(a)** Prove the policy gradient theorem: $\nabla_\theta J(\theta) = \mathbb{E}_\pi[\nabla_\theta\log\pi_\theta(a|s)\cdot Q^\pi(s,a)]$. Start from $J(\theta) = \sum_s d^\pi(s)\sum_a\pi_\theta(a|s)Q^\pi(s,a)$ where $d^\pi(s)$ is the stationary distribution.
>
> *Hint:* Use $\nabla_\theta\pi_\theta(a|s) = \pi_\theta(a|s)\nabla_\theta\log\pi_\theta(a|s)$ (the log-derivative trick).
>
> **(b)** Show that subtracting a baseline $b(s)$ from $G_t$ does not bias the gradient estimate: $\mathbb{E}_\pi[\nabla_\theta\log\pi_\theta(a|s)\cdot b(s)] = 0$.
>
> **(c)** In the RLHF objective, the KL term can be written as $D_\mathrm{KL}(\pi_\theta\|\pi_\mathrm{SFT}) = \mathbb{E}_{\pi_\theta}[\log\pi_\theta(y|x) - \log\pi_\mathrm{SFT}(y|x)]$. Interpret this: what happens when $\lambda \to 0$? When $\lambda \to \infty$?

---
# Summary

| Concept | Key formula / takeaway |
|---------|------------------------|
| **MDP** | $(\mathcal{S},\mathcal{A},P,R,\gamma)$; goal: maximise $\mathbb{E}[\sum_t\gamma^t R_t]$ |
| **Bellman equation** | $V^\pi(s) = \sum_a\pi(a\|s)[R(s,a)+\gamma\sum_{s'}P(s'\|s,a)V^\pi(s')]$ |
| **Value iteration** | Contraction $\mathcal{T}$ on $V$; converges to $V^*$ at rate $\gamma^k$ |
| **Policy gradient** | $\nabla_\theta J = \mathbb{E}[\nabla_\theta\log\pi_\theta(a\|s)\cdot G_t]$ |
| **RLHF reward model** | $P(y_1\succ y_2) = \sigma(R_\phi(x,y_1)-R_\phi(x,y_2))$ (Bradley–Terry) |
| **RLHF objective** | $\max_\theta\mathbb{E}[R_\phi(x,y)] - \lambda D_\mathrm{KL}(\pi_\theta\|\pi_\mathrm{SFT})$ |